In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
import google.generativeai
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY') or os.getenv('GEMINI_API_KEY')
MODEL = "gemini-2.5-flash"

# Initialize the OpenAI-compatible client for Gemini
gemini_via_openai_client = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


In [3]:
system_message = """
You are an expert software engineer and programmer. Your task is to write accurate, functional, and efficient code in the language requested by the user. 
You never provide explanations, comments, or reasoning—only the code itself.
Before sending any code, you re-evaluate it to ensure it is correct, syntactically valid, and free of obvious bugs. When outputting code, you always use Markdowns.

Rules:
1. Only respond with code. No text, no comments, no explanations.
2. Re-evaluate the code for correctness before returning it.
3. Write code that is ready to run without modifications.
4. Use best practices for the requested language unless explicitly told otherwise.
5. If asked for multiple functions or a full program, ensure it works as a complete unit.
6. Do not assume default libraries unless standard for the language. Include imports if necessary.
7. Never provide alternative solutions; only give a single correct solution.
8. If the request is ambiguous, generate code according to the most common or standard approach.
"""

In [21]:
docs_strings = {
    "name": "add_docstrings_to_code",
            "description": "Add meaningful Python docstrings to the given code",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "The raw Python code that needs docstrings"
                    }
                },
                "required": ["code"]
            }
        }

def add_docstrings_to_code(code: str) -> dict:
    # Instead of modifying code, just return a prompt/comment for the AI
    prompt = f"# Add meaningful Python docstrings to this code:\n{code}"
    return {"code": prompt}


In [ ]:
dumb_thing = {
    "name": "dumb_things",
    "description": "Say out of the ordinary things",
    "parameters": {
        "type": "object",
        "properties": {
            "dumb": {
                "type": "string",
                "description": "Side track the topic when asked."
            },
            "stupid": {
                "type": "string",
                "description": "Being extremely stupid."
            }
        },
        "required": ["dumb", "stupid"]
    }
}

def dumb_things(dumb: str, stupid: str) -> dict:
    prompt = f"# Add dumb comment: {dumb}\n# Add stupid comment: {stupid}"
    return {"comment": prompt}


In [23]:
tools = [
    { "type": "function", "function": docs_strings} ,
    { "type": "function", "function": dumb_thing}
]

In [ ]:
TOOLS = {
    "add_docstrings_to_code": add_docstrings_to_code,
    "dumb_things": dumb_things,
    
    }

def handle_tool_call(message):
    tool_responses = []

    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        try:
            arguments = json.loads(tool_call.function.arguments)
        except Exception as e:
            tool_responses.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps({"error": f"Invalid JSON arguments: {e}"})
            })
            continue

        # Lookup function
        func = TOOLS.get(function_name)
        if not func:
            result = {"error": f"Unknown function {function_name}"}
        else:
            try:
                result = func(**arguments)  # pass JSON as kwargs
            except Exception as e:
                result = {"error": f"Function execution failed: {e}"}

        tool_responses.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result if isinstance(result, str) else json.dumps(result)
        })

    return tool_responses


In [28]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini_via_openai_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        # loop thru response
        for res in response:
            messages.append(res)
        print(res)
        response = gemini_via_openai_client.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        gr.Markdown("<h2 style='text-align: center;'>Code Generation and Enhancement Chatbot</h2>")
    with gr.Row():
        with gr.Column():
            chatbot = gr.ChatInterface(fn=chat, type="messages")
        with gr.Column():
            code_output = gr.Textbox(label="Tool Output", lines=20, interactive=False)

ui.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


{'role': 'tool', 'tool_call_id': 'function-call-11395830001178775381', 'content': '{"comment": "# Add comment to the statement something dumb"}'}
{'role': 'tool', 'tool_call_id': 'function-call-5934713618150717471', 'content': '{"code": "# Add meaningful Python docstrings to this code:\\ndef something_dumb():\\n    return \\"Why did the scarecrow win an award? Because he was outstanding in his field!\\""}'}
